In [1]:
from pathlib import Path
import xarray as xr

In [2]:
source_dir = Path(r"E:\glacier_research\datasets\copernicus_wgms_gridded_mass_change")
output_dir = Path(r"E:\glacier_research\datasets_to_csv\copernicus_wgms_gridded_mass_change_csv")
output_dir.mkdir(parents=True, exist_ok=True)

exported_files = []

for output_year in range(1991, 2025):
    start_year = output_year - 1
    source_name = (
        f"global-gridded-annual-glacier-mass-change-"
        f"{start_year}-{output_year % 100:02d}.nc4"
    )
    source_path = source_dir / source_name

    if not source_path.exists():
        raise FileNotFoundError(f"Missing source file: {source_path}")

    with xr.open_dataset(source_path) as ds:
        nepal = ds.sel(
            lat=slice(39.3, 15.9),
            lon=slice(60.8, 105.0),
        )
        dataframe = nepal.to_dataframe().reset_index()
        dataframe = dataframe.dropna(subset=["glacier_mass_change_gt"])

    output_path = output_dir / f"nepal_glacier_data_{output_year}.csv"
    dataframe.to_csv(output_path, index=False)
    exported_files.append(output_path)

print(f"Exported {len(exported_files)} CSV files to {output_dir}")
print(f"Years: {exported_files[0].stem.rsplit('_', 1)[-1]}-"
      f"{exported_files[-1].stem.rsplit('_', 1)[-1]}")

Exported 34 CSV files to E:\glacier_research\datasets_to_csv\copernicus_wgms_gridded_mass_change_csv
Years: 1991-2024


In [3]:
import pandas as pd

csv_dir = Path(r"E:\glacier_research\datasets_to_csv\copernicus_wgms_gridded_mass_change_csv")
input_files = [
    csv_dir / f"nepal_glacier_data_{year}.csv"
    for year in range(1991, 2025)
]
missing_files = [path for path in input_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        "Missing yearly CSV files: "
        + ", ".join(path.name for path in missing_files)
    )

combined_data = pd.concat(
    (pd.read_csv(path) for path in input_files),
    ignore_index=True,
)
combined_path = Path(
    r"E:\glacier_research\datasets_to_csv\copernicus_wgms_gridded_mass_change_csv\nepal_glacier_data_1990-2024.csv"
)
combined_data.to_csv(combined_path, index=False)

print(f"Combined {len(input_files)} yearly files")
print(f"Total rows: {len(combined_data)}")
print(f"Saved to: {combined_path}")

Combined 34 yearly files
Total rows: 23698
Saved to: E:\glacier_research\datasets_to_csv\copernicus_wgms_gridded_mass_change_csv\nepal_glacier_data_1990-2024.csv
